In [0]:
import pandas as pd
from datetime import datetime
import json

# Simulated live global energy news stream to prevent 404 network errors
raw_rss_mock_data = [
    {
        "title": "Global Crude Prices Stabilize Amid Supply Adjustments",
        "link": "https://energy-news-hub.com/article1",
        "pubDate": "Mon, 06 Jul 2026 08:00:00 GMT",
        "description": "Crude oil benchmarks held steady this morning as market analysts weighed OPEC production shifts against rising industrial demand in Asia."
    },
    {
        "title": "Offshore Wind Project Greenlit for North Sea Expansion",
        "link": "https://energy-news-hub.com/article2",
        "pubDate": "Mon, 06 Jul 2026 09:15:00 GMT",
        "description": "A major consortium secured regulatory approvals to deploy an additional 1.2 gigawatts of capacity, accelerating regional transition goals."
    },
    {
        "title": "Natural Gas Futures Undergo Seasonal Volatility",
        "link": "https://energy-news-hub.com/article3",
        "pubDate": "Mon, 06 Jul 2026 10:30:00 GMT",
        "description": "Storage injections falling slightly below the five-year average triggered brief trading spikes, driving utility operators to adjust reserves."
    }
]

try:
    articles = []
    # Loop through the feed items just like parsing an XML structure
    for item in raw_rss_mock_data:
        title = item.get('title', '')
        link = item.get('link', '')
        pub_date = item.get('pubDate', '')
        description = item.get('description', '')
        
        articles.append({
            "title": title, 
            "link": link, 
            "published": pub_date, 
            "raw_text": description
        })
        
    # Convert to a standard pandas DataFrame
    df = pd.DataFrame(articles)
    print(f"Successfully grabbed {len(df)} raw energy articles!")
    display(df.head())
    
except Exception as e:
    print(f"Error processing data: {e}")

In [0]:
# --- SILVER LAYER PROCESSING ---
# Cleaning, structuring, and standardizing the raw text

# 1. Ensure no exact duplicate titles exist
df_silver = df.drop_duplicates(subset=['title']).copy()

# 2. Standardize timestamps to standard ISO format for data consistency
df_silver['published_parsed'] = pd.to_datetime(df_silver['published'], errors='coerce')

# 3. Clean up the raw text (strip whitespace, ensure string type)
df_silver['clean_text'] = df_silver['raw_text'].astype(str).str.strip()

# 4. Drop the old raw columns we no longer need to keep the schema tight
df_silver = df_silver[['title', 'link', 'published_parsed', 'clean_text']]

print(f"Silver Layer complete: Optimized {len(df_silver)} clean articles.")
display(df_silver.head())

In [0]:
# --- GOLD LAYER PROCESSING (LLM OPTIMIZATION) ---
# Transforming clean text into math vectors for LLM consumption

import hashlib
import numpy as np

def generate_mock_embedding(text, dimensions=8):
    """
    Simulates an embedding model by turning text into a repeatable 
    array of floating-point numbers representing semantic meaning.
    """
    # Create a stable hash from the text string
    hash_object = hashlib.sha256(text.encode('utf-8'))
    hash_hex = hash_object.hexdigest()
    
    # Generate numbers between -1.0 and 1.0 based on the hash pieces
    np.random.seed(int(hash_hex[:8], 16) % (2**32 - 1))
    vector = np.random.uniform(-1.0, 1.0, dimensions).round(4).tolist()
    return vector

# 1. Create a copy for our Gold presentation tier
df_gold = df_silver.copy()

# 2. Generate the Vector Embeddings column from our clean text
df_gold['vector_embedding'] = df_gold['clean_text'].apply(generate_mock_embedding)

# 3. Select final columns optimized for vector database ingestion
df_gold = df_gold[['title', 'clean_text', 'vector_embedding']]

print("Gold Layer complete: Vector database payload ready for LLM consumption.")
display(df_gold.head())

In [0]:
# --- WRITING TO THE GOLD DELTA LAKE TABLE ---
# Saving the engineered payload so a model can query it

try:
    # Convert the pandas DataFrame to a Spark DataFrame for native Databricks storage
    spark_df_gold = spark.createDataFrame(df_gold)
    
    # Save the data as a managed Delta table in your lakehouse
    # This physically writes the text and vectors to disk
    spark_df_gold.write.format("delta").mode("overwrite").saveAsTable("gold_energy_embeddings")
    
    print("[SUCCESS] Gold Delta table created! A model can now query this dataset.")
    
    # Verify by reading it back from the system
    display(spark.table("gold_energy_embeddings"))
    
except Exception as e:
    print(f"[ERROR] Could not write to table: {e}")